# Digital Identity and Emotional Coping Among Young Adults on Instagram and TikTok
### Reproducibility Notebook

**Author:** Nasir Hussain, Assistant Registrar, Karachi Institute of Economics and Technology (KIET)

This notebook reproduces, in order, all statistical analyses reported in the manuscript *"Digital Identity and Emotional Coping Among Young Adults on Instagram and TikTok."* It clones the study's GitHub repository, loads the anonymized survey dataset, prepares the variables, and reproduces:

1. Reliability analysis (Cronbach's alpha)
2. Descriptive statistics
3. Pearson correlation analysis
4. Regression analysis
5. Baron & Kenny mediation analysis with the Sobel test
6. Group comparisons (gender, age, usage intensity, platform)

Run all cells top to bottom (**Runtime → Run all**) in Google Colab to reproduce every table reported in the manuscript.

**Repository:** https://github.com/nasirhussain92/social-media-digital-identity-jmh


## 1. Setup
Clone the repository and install dependencies.

In [ ]:
# Clone the reproducibility repository (skips if already present, e.g. when run from within the repo)
import os

REPO_URL = "https://github.com/nasirhussain92/social-media-digital-identity-jmh.git"
REPO_DIR = "social-media-digital-identity-jmh"

if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL}

%cd {REPO_DIR}


In [ ]:
# Install dependencies
!pip install -q -r requirements.txt


In [ ]:
import sys
sys.path.append("src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from utils import load_dataset, reverse_score, create_composite, missing_values, dataset_summary
from reliability import reliability_report
from descriptive import descriptive_statistics, export_descriptive
from correlation import correlation_matrix, correlation_pvalues, export_correlation
from regression import run_regression, regression_table, model_statistics, export_regression
from mediation import mediation_analysis, export_mediation

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 120)

import os
os.makedirs("outputs/tables", exist_ok=True)
os.makedirs("outputs/figures", exist_ok=True)


## 2. Load Dataset

In [ ]:
df_raw = load_dataset("data/social_media_dataset.csv")
codebook = pd.read_csv("data/codebook.csv")

dataset_summary(df_raw)
df_raw.head()


In [ ]:
codebook

## 3. Data Preparation

Following the manuscript's inclusion criterion, only respondents who provided voluntary informed consent (`Consent == "Yes"`) are retained, yielding the final analytical sample of **N = 265**. Missing values were checked and none were found in the anonymized dataset.

In [ ]:
print("Before consent filter:", df_raw.shape)
print(df_raw["Consent"].value_counts())

df = df_raw[df_raw["Consent"] == "Yes"].reset_index(drop=True)

print("Final analytical sample:", df.shape)
missing_values(df)


### 3.1 Reverse Scoring

Five Rosenberg Self-Esteem Scale items are reverse-worded and reverse-scored using the formula `6 − raw score` (i.e., `(max_scale + min_scale) − raw score` for a 1–5 scale), per the codebook's `Reverse_Scored` flag.

In [ ]:
reverse_items = codebook.loc[codebook["Reverse_Scored"] == "Yes", "Variable"].tolist()
print("Reverse-scored items:", reverse_items)

df = reverse_score(df, reverse_items, max_scale=5, min_scale=1)


### 3.2 Composite Variable Construction

Composite scores are computed as the arithmetic mean of each construct's items:
- **SMU** — Social Media Usage Intensity (SMU1–SMU5)
- **DI** — Digital Identity Expression (DI1–DI5)
- **EC** — Emotional Coping (EC1–EC4)
- **SC** — Social Comparison (SC1–SC5)
- **SE** — Self-Esteem, Rosenberg Scale (SE1–SE10, reverse-scored items included)

In [ ]:
df = create_composite(df, ["SMU1","SMU2","SMU3","SMU4","SMU5"], "SMU")
df = create_composite(df, ["DI1","DI2","DI3","DI4","DI5"], "DI")
df = create_composite(df, ["EC1","EC2","EC3","EC4"], "EC")
df = create_composite(df, ["SC1","SC2","SC3","SC4","SC5"], "SC")
df = create_composite(df, ["SE1","SE2","SE3","SE4","SE5","SE6","SE7","SE8","SE9","SE10"], "SE")

df[["SMU","DI","EC","SC","SE"]].head()


## 4. Reliability Analysis (Cronbach's Alpha)

Reproduces Table 1 (reliability column). Reported manuscript values: SMU α = .824, DI α = .789, EC α = .833, SC α = .899, SE α = .725 (all within the .725–.899 range noted in the abstract).

In [ ]:
reliability = reliability_report(df)
reliability


## 5. Descriptive Statistics

Reproduces Table 1 (descriptive columns).

In [ ]:
# Note: export_* functions default to a "../outputs/tables/..." path (designed to be called
# from within src/ or notebooks/). Since this notebook's cwd is the repo root after cloning,
# we pass explicit "outputs/tables/..." paths here.
desc = descriptive_statistics(df)
export_descriptive(desc, filename="outputs/tables/descriptive_statistics.csv")
desc


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)
for ax, var in zip(axes, ["SMU", "DI", "EC", "SC", "SE"]):
    sns.histplot(df[var], kde=True, ax=ax, color="#4C72B0")
    ax.set_title(var)
plt.tight_layout()
plt.savefig("outputs/figures/distributions.png", dpi=150)
plt.show()


## 6. Correlation Analysis

Reproduces Table 2 (Pearson correlation matrix). The manuscript highlights Social Comparison as the strongest correlate of Self-Esteem (r = −.424, p < .001).

In [ ]:
corr = correlation_matrix(df)
export_correlation(corr, filename="outputs/tables/correlation_matrix.csv")
corr


In [ ]:
pvals = correlation_pvalues(df)
pvals


In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, center=0)
plt.title("Pearson Correlation Matrix")
plt.tight_layout()
plt.savefig("outputs/figures/correlation_heatmap.png", dpi=150)
plt.show()


## 7. Regression Analysis

Simple regressions predicting Self-Esteem from each predictor, reproducing the manuscript's regression section (Social Comparison as the dominant predictor: β = −.263, R² = .180, p < .001).

In [ ]:
predictors = ["SMU", "DI", "EC", "SC"]
regression_summaries = []

for pred in predictors:
    model = run_regression(df, "SE", [pred])
    stats = model_statistics(model)
    stats.insert(0, "Predictor", pred)
    regression_summaries.append(stats)
    print(f"--- Self-Esteem ~ {pred} ---")
    print(regression_table(model))
    print(stats.to_string(index=False))
    print()

regression_summary_table = pd.concat(regression_summaries, ignore_index=True)
regression_summary_table


In [ ]:
# Save the Social Comparison model (the dominant predictor) as the primary regression table
sc_model = run_regression(df, "SE", ["SC"])
export_regression(regression_table(sc_model), filename="outputs/tables/regression_results.csv")
regression_table(sc_model)


## 8. Mediation Analysis (Baron & Kenny + Sobel Test)

Two parallel mediation models are reproduced, both with Social Comparison (SC) as the mediator and Self-Esteem (SE) as the outcome:

- **Model 1:** Digital Identity Expression (DI) → Social Comparison (SC) → Self-Esteem (SE)
- **Model 2:** Emotional Coping (EC) → Social Comparison (SC) → Self-Esteem (SE)

Manuscript-reported results: Model 1 indirect effect = −0.197, Sobel z = −6.41, p < .001; Model 2 indirect effect = −0.138, Sobel z = −5.28, p < .001.

In [ ]:
mediation_di = mediation_analysis(df, independent="DI", mediator="SC", dependent="SE")
mediation_di["Indirect effect"] = (mediation_di["Path a"] * mediation_di["Path b"]).round(3)
mediation_di.insert(0, "Model", "DI -> SC -> SE")
mediation_di


In [ ]:
mediation_ec = mediation_analysis(df, independent="EC", mediator="SC", dependent="SE")
mediation_ec["Indirect effect"] = (mediation_ec["Path a"] * mediation_ec["Path b"]).round(3)
mediation_ec.insert(0, "Model", "EC -> SC -> SE")
mediation_ec


In [ ]:
mediation_summary = pd.concat([mediation_di, mediation_ec], ignore_index=True)
export_mediation(mediation_summary, filename="outputs/tables/mediation_results.csv")
mediation_summary


## 9. Group Comparisons

Reproduces the manuscript's descriptive group comparisons by gender, age cohort, daily usage intensity, and preferred platform.

In [ ]:
gender_summary = df.groupby("Gender")[["SC", "SE"]].mean().round(3)
gender_summary


In [ ]:
age_summary = df.groupby("Age")[["SC", "SE"]].mean().round(3)
age_summary


In [ ]:
usage_summary = df.groupby("DailyUsage")[["SC", "SE"]].mean().round(3)
usage_summary


In [ ]:
platform_summary = df.groupby("Platform")[["SC", "SE"]].mean().round(3)
platform_summary


In [ ]:
group_tables = {
    "gender": gender_summary,
    "age": age_summary,
    "usage": usage_summary,
    "platform": platform_summary,
}
for name, table in group_tables.items():
    table.to_csv(f"outputs/tables/group_comparison_{name}.csv")

print("Group comparison tables saved to outputs/tables/")


## 10. Summary

All tables reproduced in this notebook are written to `outputs/tables/` and all figures to `outputs/figures/`, matching the repository structure described in `README.md`:

- `descriptive_statistics.csv`
- `correlation_matrix.csv`
- `regression_results.csv`
- `mediation_results.csv`
- `group_comparison_*.csv`

This notebook, together with the `src/` scripts and the anonymized `data/` files, constitutes the full reproducibility package for the manuscript *"Digital Identity and Emotional Coping Among Young Adults on Instagram and TikTok."* When archived via Zenodo, cite the repository using the details in `CITATION.cff`.